In [1]:
import pandas as pd 
import numpy as np


In [2]:
import sys
print(sys.executable)


/home/ishido/Desktop/Aj/New proj/HELP extend/environment/venv/bin/python


In [3]:
PPI_edges = pd.read_csv('../../../data/kidney/processing/kidney_PPI_final.csv')

In [4]:
labels = pd.read_csv('../../../data/kidney/processing/Crispr_labels_final.csv')

In [5]:
expressions = pd.read_csv('../../../data/kidney/processing/Kidney_expression_final.csv')

In [6]:
mutations = pd.read_csv('../../../data/kidney/processing/mutations_final.csv')

In [7]:
common = set(expressions['ModelID']) & set(labels['ModelID']) & set(mutations['ModelID'])


In [ ]:
len(common)



In [9]:
expressions = expressions[expressions['ModelID'].isin(common)]
mutations = mutations[mutations['ModelID'].isin(common)]
labels = labels[labels['ModelID'].isin(common)]


In [ ]:
expressions.shape

In [ ]:
mutations.shape

In [ ]:
labels.shape

In [13]:
expressions = expressions.sort_values('ModelID').reset_index(drop=True)
mutations   = mutations.sort_values('ModelID').reset_index(drop=True)
labels      = labels.sort_values('ModelID').reset_index(drop=True)

cell_line_ids = expressions['ModelID'].values

assert list(expressions['ModelID']) == list(mutations['ModelID']) == list(labels['ModelID']), \
    "ModelID order mismatch — positional indexing downstream will silently mix cell lines"


In [ ]:
expressions.head()

In [ ]:
mutations.head()

In [ ]:
labels.head()

In [17]:
expressions.to_csv("../../../data/kidney/expressions.csv",index=False)

In [18]:
mutations.to_csv("../../../data/kidney/mutations.csv",index=False)

In [19]:
labels.to_csv("../../../data/kidney/labels.csv",index = False)

In [20]:
exprxn = pd.read_csv("../../../data/kidney/processing/Kidney_expression_final.csv")

In [ ]:
exprxn

In [ ]:
ppi = pd.read_csv("../../../data/kidney/processing/kidney_PPI_final.csv")
ppi

In [23]:
exprxn_values = exprxn.drop('ModelID',axis=1)

In [ ]:
exprxn_values

In [25]:
A_values = exprxn_values[ppi['A']].values
B_values = exprxn_values[ppi['B']].values

In [ ]:
A_values

In [ ]:
A_values.nbytes / 1e9


In [28]:
A_Centered = A_values - A_values.mean(axis=0)
B_Centered = B_values - B_values.mean(axis=0)

In [ ]:
A_Centered

In [30]:
num = (A_Centered * B_Centered).sum(axis=0)

In [31]:
denom = np.sqrt((A_Centered**2).sum(axis=0) * (B_Centered**2).sum(axis=0))


In [ ]:
coexpression = num / denom
coexpression[10:]


In [ ]:
np.isnan(coexpression).sum()


In [34]:
coexpression = np.nan_to_num(coexpression, nan=0.0)


In [ ]:
coexpression.shape


In [ ]:
np.isnan(coexpression).sum()


In [ ]:
ppi['coexpression'] = coexpression
ppi.head()

In [38]:
ppi.to_csv('../../../data/kidney/ppi.csv', index=False)
